# 08 Security and Governance Guardrails (LiteLLM, 2026)

## What This Lesson Is
Enforce model allowlists, prompt redaction, and policy checks before outbound model execution.

## Scientific Lens
- Concept: Policy-as-code guardrails for LLM access
- Measure: Blocked policy violations and redaction coverage
- Validity Limit: Regex-based controls must be complemented by layered DLP and audit controls.


## How It Works
1. Define explicit guardrail policy.
2. Run deterministic policy checks.
3. Gate a live call through policy enforcement.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Allowed models: openai/gpt-4.1-mini, openai/gpt-4o-mini")


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
import re

allowed_models = {"openai/gpt-4.1-mini", "openai/gpt-4o-mini"}
request = {
    "model": "openai/gpt-4.1-mini",
    "prompt": "Deploy using token sk-live-abcdef123 and report status.",
}

if request["model"] not in allowed_models:
    raise PermissionError("model blocked")

safe_prompt = re.sub(r"sk-[A-Za-z0-9-]+", "[REDACTED_TOKEN]", request["prompt"])
print(safe_prompt)
assert "sk-" not in safe_prompt


In [ ]:
# Live Demo
import os
import re

try:
    from litellm import completion
except Exception as exc:
    print(f"Skipping live guardrail demo: litellm unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    model = "openai/gpt-4.1-mini"
    allowed_models = {"openai/gpt-4.1-mini", "openai/gpt-4o-mini"}

    if model not in allowed_models:
        print("Blocked by allowlist policy")
    elif not api_key:
        print("Skipping live guardrail demo: OPENAI_API_KEY not set.")
    else:
        prompt = "Summarize governance controls for LLM systems in one sentence."
        prompt = re.sub(r"sk-[A-Za-z0-9-]+", "[REDACTED_TOKEN]", prompt)
        r = completion(model=model, messages=[{"role": "user", "content": prompt}], api_key=api_key, timeout=20)
        print(r.choices[0].message.content.strip())


## Applied Labs
1. Add per-team model allowlists and verify cross-team access boundaries.
2. Block prompts containing banned domains and test policy exceptions.
3. Log policy decision records with timestamp, reason, and policy ID.

## Validation Checklist
- Model execution is gated by explicit allowlist.
- Sensitive token patterns are redacted before outbound requests.
- Policy decisions are transparent and auditable.

## Further Reading
- [OWASP LLM Top 10](https://owasp.org/www-project-top-10-for-large-language-model-applications/)
- [NIST AI RMF](https://www.nist.gov/itl/ai-risk-management-framework)
- [LiteLLM Security Guidance](https://docs.litellm.ai/docs/proxy/security)
